# Can decision-model activations improve guardrail decisions?
## A Colab experiment for raxIT Labs — Kev 4B

This adapts Aditya's ARENA truth-probing workflow to **block / allow under a fixed policy**.
We load the **trained Kev checkpoint, including its LoRA and decision head**. We do not substitute its Qwen base.

The question is: **Do intermediate activations contain useful policy-violation information that a supervised linear readout can recover better than the model's native head?**

Run on a **fresh Colab GPU runtime**. A100 is recommended for speed; start with a T4 if necessary. The 4B backbone needs more memory than Strands 2B; an A100 is a practical choice for this run. Weights alone do not determine peak VRAM: question batches and sequence lengths also matter. No paid inference APIs or API keys are needed. Colab compute still uses your credits.

**First run:** install cell → restart session → run from cell 2 onward. `SMOKE=True` uses a small grouped sample to check the entire pipeline. For research results, set `SMOKE=False` before extraction, then run the analysis again from a fresh output directory. Never report the smoke run as a benchmark result.

Graphs appear only after inference: layer sweep, PCA, test ROC, confusion matrices, performance with group-bootstrap intervals, per-topic scores, and order-sensitivity plots. All graphs and results are exported in a ZIP.

**Validation status:** notebook Python was syntax-checked; data splitting, probe fitting, scoring, and graph generation were checked locally with real metadata and synthetic feature matrices. The native loader and extraction APIs were checked against the pinned Kev source. The pretrained GPU forward pass was not run in the authoring environment. Results are intentionally empty until you run Colab.

### Models on the board with downloadable weights (checked 13 October 2026)

| Board system | Public artifact | Practical first experiment |
|---|---|---|
| Strands Decider 2B | `StrandsAgents/strands-decider-2B-hobson-v19` | This notebook; native loader plus layer hooks |
| Kev 0.8B / 4B / 9B | `jaredpalmer/kev-0.8b`, `kev-4b`, `kev-9b` | Native Kev loader; adapters and separate head |
| Open-Jev 2B | `ZefanCai/Open-Jev-2B` | Native Open-Jev loader; adapters and separate head |
| Laya | `convaiinnovations/laya` | Small encoder; option-marker / masked pooling, not a causal last-token recipe |
| Clef Flash / Clef | `Cloudflare/clef-flash`, `Cloudflare/clef` | 9B / 27B; native joint-schema head |
| Decisions API · Perplexity | `perplexity-ai/pplx-decider-v1-27b` | 27B; native loader; vendor documents ~49 GiB weights plus working memory |

Jev and OpenAI's hosted Decisions API do not expose the internal activations required here. A hosted API can serve a model whose weights are also public; its HTTP response still does not provide hidden states. Local results need not exactly match hosted serving.

Primary sources:
- [raxIT board](https://dm-as-guardrails.raxitlabs.com/), [code](https://github.com/raxITlabs/decision-models-as-guardrails), [dataset](https://huggingface.co/datasets/raxITLabs/decision-models-as-guardrails)
- [Strands model card](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19), [implementation](https://github.com/strands-labs/strands-decider)
- [Kev](https://github.com/jaredpalmer/kev), [Open-Jev](https://huggingface.co/ZefanCai/Open-Jev-2B), [Laya](https://huggingface.co/convaiinnovations/laya)
- [Clef](https://huggingface.co/Cloudflare/clef), [Clef Flash](https://huggingface.co/Cloudflare/clef-flash), [Perplexity](https://huggingface.co/perplexity-ai/pplx-decider-v1-27b), [TypeSafe access](https://docs.typesafe.ai/models)

### 1. Install once, then restart the Colab session

Use a fresh runtime rather than the ARENA environment; its Transformers 4.56 pin does not support this loader.
Install Jared Palmer’s Kev from GitHub: the PyPI package called `kev` is an unrelated ORM. The repository code, model adapter/head, dataset, and policy questions are pinned below. Dependency versions are recorded in the run manifest. Optional fast kernels can make hybrid Qwen layers faster, but the default uses the reference implementation to avoid a CUDA compilation requirement.

In [ ]:
import subprocess, sys
KEV_CODE_REV = "85ccf56a0afb952323c621055b02876e8498d60c"
# Install the correct Kev project, not the unrelated PyPI ORM.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    f"git+https://github.com/jaredpalmer/kev.git@{KEV_CODE_REV}",
    "pandas>=2.2,<3", "matplotlib>=3.8,<4", "tqdm>=4.66"])
# PEFT rejects Colab's older optional torchao; this experiment does not use it.
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
print("Installation complete. Use Runtime > Restart session, then run from configuration onward.")

### 2. Configuration and evaluation contract

`expected=yes` means **block**; `expected=no` means **allow**. The policy, not universal factual truth or intrinsic harmfulness, defines the label. Some benign text is blocked under a denied-topic policy.

We use the topic suite because its pinned release has 118 dev and 540 test rows with readable text. Many other suites ship mainly IDs and need reconstruction from upstream sources. Missing text is counted and excluded, never filled with labels or IDs.

Training and validation come only from the published `dev` split. Related pairs stay together using `group`. Layer, pooling, LR regularization and threshold are selected on validation; test is used only afterward. This is a **new supervised probing experiment on a readable subset**, not a reproduction of the board's unsupervised default ranking or its overall score.

In [ ]:
import os
os.environ["USE_TF"] = "0"
import json, hashlib, platform, time, copy, random, zipfile, importlib.metadata as im
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
import torch
from huggingface_hub import hf_hub_download, snapshot_download
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics import (roc_auc_score, roc_curve, balanced_accuracy_score,
    accuracy_score, confusion_matrix, average_precision_score, brier_score_loss)
from kev.checkpoint import Checkpoint, LoadOptions
from kev.api import SystemOneRequest, to_record
from kev.model import rows_of

SEED = 42
SMOKE = False
QUESTION_BATCH = 1
RUN_ROBUSTNESS = True
QUESTION_BATCH = 2           # Independent question sequences per forward; 1 if OOM.
N_BOOT = 500                 # Increase to 2000 for the final report.
MODEL_ID = "jaredpalmer/kev-4b"
MODEL_REV = "6cfce5c2fa4b4bd64026336ab649c5ca78857d52"  # v1.0 resolved to a fixed commit
KEV_CODE_REV = "85ccf56a0afb952323c621055b02876e8498d60c"
BENCHMARK_REV = "38b70dc2991e52b62b1e05def887e268e5ac4a7c"
DATA_ID = "raxITLabs/decision-models-as-guardrails"
DATA_REV = "13beb711a4c2de3f9652bab02570c7c439a2f79b"
OUT = Path("/content/raxit_kev4b_probe_smoke" if SMOKE else "/content/raxit_kev4b_probe_full")
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select a GPU in Colab: Runtime > Change runtime type."
DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| dtype:", DTYPE, "| smoke:", SMOKE)
print("No network calls to a paid model endpoint are made by this notebook.")

def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()

### 3. Load rows and the exact frozen policy questions

Only `state` is sent to the model. Expected labels, rationales, proposed labels, provenance, source IDs and topic tags never enter the prompt. Topic tags are used only for the later evaluation breakdown.

In [ ]:
import urllib.request
qurl = (f"https://raw.githubusercontent.com/raxITlabs/decision-models-as-guardrails/"
        f"{BENCHMARK_REV}/benchmark/question_sets/e2/f3-topics.json")
with urllib.request.urlopen(qurl, timeout=60) as response:
    QSET = json.load(response)
QKEYS = list(QSET["decision"])
assert all(QSET["questions"][k]["type"] == "noul" for k in QKEYS)
(OUT / "frozen_questions.json").write_text(json.dumps(QSET, indent=2))

row_audit = []
def load_rows(split):
    path = hf_hub_download(DATA_ID, f"data/denied_topics/{split}.jsonl",
                          repo_type="dataset", revision=DATA_REV)
    raw = [json.loads(s) for s in Path(path).read_text().splitlines() if s.strip()]
    rows = []
    for r in raw:
        text = (r.get("state") or {}).get("text")
        if not isinstance(text, str) or not text.strip():
            continue
        if r.get("expected") not in ("yes", "no"):
            continue
        # Exact state fields preserved; no metadata is concatenated into it.
        state = dict(r["state"])
        fingerprint = hashlib.sha256(json.dumps(state, sort_keys=True,
            ensure_ascii=False).encode()).hexdigest()
        rows.append(dict(id=r["id"], group=r.get("group") or r["id"],
            y=int(r["expected"] == "yes"), state=state,
            topic=(r.get("attribute") or {}).get("topic") or "Other",
            source=(r.get("provenance") or {}).get("source", "unknown"),
            state_hash=fingerprint))
    row_audit.append(dict(split=split, total=len(raw), readable=len(rows),
                          omitted=len(raw)-len(rows)))
    return pd.DataFrame(rows)

dev = load_rows("dev")
# Collapse exact duplicate model inputs, refusing contradictory labels.
def dedupe(df):
    assert df.groupby("state_hash").y.nunique().max() == 1, "Conflicting labels for identical input."
    return df.drop_duplicates("state_hash").reset_index(drop=True)
dev = dedupe(dev)
# Grouped split is fixed before any activations or performance are inspected.
for seed in range(SEED, SEED + 100):
    tr, va = next(GroupShuffleSplit(n_splits=1, test_size=.25,
                                   random_state=seed).split(dev, dev.y, dev.group))
    if dev.iloc[tr].y.nunique() == 2 and dev.iloc[va].y.nunique() == 2:
        break
else:
    raise ValueError("Could not create a two-class grouped validation split.")
train = dev.iloc[tr].reset_index(drop=True)
valid = dev.iloc[va].reset_index(drop=True)

def small_group_sample(df, n_groups):
    # Include whole groups; keep sampling if needed to obtain both classes.
    groups = np.random.default_rng(SEED).permutation(df.group.unique())
    for n in range(min(n_groups, len(groups)), len(groups)+1):
        sampled = df[df.group.isin(groups[:n])].reset_index(drop=True)
        if sampled.y.nunique() == 2:
            return sampled
    raise ValueError("Need both classes.")
if SMOKE:
    train = small_group_sample(train, 12)
    valid = small_group_sample(valid, 6)
assert set(train.group).isdisjoint(set(valid.group))
# Test rows are not loaded until hyperparameters have been frozen.
display(pd.DataFrame(row_audit))
display(pd.DataFrame([dict(split=n, rows=len(d), groups=d.group.nunique(),
    block=int(d.y.sum()), allow=int((1-d.y).sum())) for n,d in [("train",train),("validation",valid)]]))
print("Decision questions:", QKEYS)

In [ ]:
%pip uninstall -y torchaudio torchvision torchao

### 4. Load the trained adapter + native decision head

Kev’s checkpoint metadata pins its upstream Qwen revision when available. No quantization is used. On T4, the torso is cast to fp16; on supported GPUs it stays bf16. The small trained head stays fp32. Record this precision when comparing to published numbers.

The native loader must be used: a generic Qwen `AutoModelForCausalLM` call would omit the trained decision adapter/head.

In [ ]:
DTYPE = torch.float16  # Half precision for your current GPU

checkpoint = Checkpoint(f"{MODEL_ID}@{MODEL_REV}")

# Complete adapter loading and precision conversion on CPU.
tokenizer, model = checkpoint.load("cpu", LoadOptions(
    dtype=DTYPE, merge=False, backend="torch", attn="eager",
    cuda_graphs=False, fused=False,
))

model.lm.to(dtype=DTYPE)
model.head.to(dtype=torch.float32)

# Transfer the finished model to GPU once.
model.to(DEVICE)
model.device = DEVICE
model.requires_grad_(False)
model.eval()

print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 2**30, 2), "GiB")
# Keep the published LoRA unmerged, and leave the small decision head in fp32.
base = model.lm.get_base_model() if hasattr(model.lm, "get_base_model") else model.lm
layers = base.layers
LAYERS = sorted(set(np.linspace(0, len(layers)-1, 7, dtype=int).tolist()))
TRANSFER_LAYER = LAYERS[len(LAYERS)//2]
MAX_STATE_TOKENS = 4096
MAX_ROW_TOKENS = 4096
print("Checkpoint:", MODEL_ID, MODEL_REV)
print("Base:", checkpoint.meta.base, "revision:", checkpoint.meta.base_revision)
print("Decoder blocks:", len(layers), "sampled blocks:", LAYERS)
print("Pointer-head temperature:", model.head.temperature)
print("LoRA merge / fused kernels / CUDA graphs: disabled for observable layer hooks.")
if checkpoint.meta.base_revision is None:
    print("Upstream revision is unpinned; record this reproducibility limitation.")

### 5. Read intermediate states during the native decision computation

Like your notebook, this gets a vector from each chosen layer. The difference is **where**:
- `answer_mean`: `<decide>` token, averaged over the frozen policy questions.
- `option_delta`: closing marker of the true option minus that of the false option, averaged over questions.

These pooled features are linear functions of the captured activations. The downstream MM / LR readouts are linear in them. We sample raw outputs of numbered decoder blocks, **before the final decoder normalization**; this is explicit and can differ from `outputs.hidden_states[-1]`.

The native baseline is **max P(true) across the nine frozen decision questions**, with Kev’s saved head temperature. Kev uses its own state rendering, `<decide>` and closing-option markers. Each Qwen3.5 question runs as an independent causal row; the notebook preserves that native isolation. Hooks capture only requested positions and move small vectors to CPU. Prefix caching is disabled so each position belongs to a full prompt. One row is processed at a time; `QUESTION_BATCH` controls the question batch.

In [ ]:
FEATURES = ("answer_mean", "option_delta")

def kev_record(state, variant="original"):
    if variant == "json_reorder":
        state = dict(reversed(list(state.items())))
    request = SystemOneRequest(state=state,
        questions={k: QSET["questions"][k] for k in QKEYS})
    record, metadata = to_record(request)
    if variant == "option_reorder":
        # Change order while preserving the meaning of each option and its score mapping.
        for q, m in zip(record["questions"], metadata):
            q["options"] = list(reversed(q["options"]))
            m["keys"] = list(reversed(m["keys"]))
    return record, metadata


def predict_and_capture(state, requested_layers, variant="original"):
    record, metadata = kev_record(state, variant)
    enc = model.encode(tokenizer, record, max_state=MAX_STATE_TOKENS,
                       max_branch=MAX_ROW_TOKENS, strict=True)
    assert not enc["state_truncated"]
    state_ids, state_pos, branches = rows_of(enc)
    collected = {f: {l: [] for l in requested_layers} for f in FEATURES}
    native_per_q, lengths = [], []
    for start in range(0, len(branches), QUESTION_BATCH):
        part = branches[start:start+QUESTION_BATCH]
        meta = metadata[start:start+QUESTION_BATCH]
        causal_rows = [(state_ids + r["ids"], state_pos + r["pos"]) for r in part]
        ids, positions, mask = model._pad_rows(causal_rows)
        bi = torch.arange(len(part), device=DEVICE)
        decide = torch.tensor([len(state_ids)+r["decide"] for r in part], device=DEVICE)
        option_positions = [torch.tensor([len(state_ids)+o for o in r["opts"]],
                                       device=DEVICE) for r in part]
        true_pos = torch.tensor([int(p[m["keys"].index("true")])
                                 for p,m in zip(option_positions,meta)], device=DEVICE)
        false_pos = torch.tensor([int(p[m["keys"].index("false")])
                                  for p,m in zip(option_positions,meta)], device=DEVICE)
        handles = []
        def make_hook(layer_index):
            def hook(_module, _inputs, output):
                h = output[0] if isinstance(output,(tuple,list)) else output
                assert h.ndim == 3 and h.shape[:2] == ids.shape
                collected["answer_mean"][layer_index].append(h[bi,decide].float().cpu().numpy())
                collected["option_delta"][layer_index].append(
                    (h[bi,true_pos].float()-h[bi,false_pos].float()).cpu().numpy())
            return hook
        try:
            for l in requested_layers:
                handles.append(layers[l].register_forward_hook(make_hook(l)))
            with torch.inference_mode():
                hidden = model.lm(input_ids=ids, position_ids=positions,
                    attention_mask=mask, use_cache=False).last_hidden_state.float()
                for b,(opts,m) in enumerate(zip(option_positions,meta)):
                    # model.head applies its saved calibration temperature in eval mode.
                    logits = model.head(hidden[b,decide[b]],hidden[b,opts])
                    probabilities = torch.softmax(logits.float(),dim=-1)
                    native_per_q.append(float(probabilities[m["keys"].index("true")].cpu()))
            lengths.extend(mask.sum(-1).cpu().tolist())
        finally:
            for handle in handles: handle.remove()
    feats = {f: {l: np.concatenate(collected[f][l]).mean(axis=0)
                 for l in requested_layers} for f in FEATURES}
    assert len(native_per_q) == len(QKEYS)
    assert all(np.isfinite(v).all() for by_layer in feats.values() for v in by_layer.values())
    return feats, float(max(native_per_q)), native_per_q, max(lengths)

# Compare the instrumented path with Kev's own native, uncached forward/readout.
# Reduced-precision batch shapes can move probabilities slightly; this is a numerical check.
first_state = train.iloc[0].state
_, smoke_score, smoke_q, _ = predict_and_capture(first_state, LAYERS)
record, metadata = kev_record(first_state)
enc = model.encode(tokenizer, record, max_state=MAX_STATE_TOKENS,
                   max_branch=MAX_ROW_TOKENS, strict=True)
with torch.inference_mode():
    reference_logits = model.forward(enc)
reference_q = [float(torch.softmax(z.float(),-1)[m["keys"].index("true")].cpu())
               for z,m in zip(reference_logits,metadata)]
np.testing.assert_allclose(smoke_q,reference_q,atol=2e-2,rtol=0)
print("Native-path numerical parity passed; max score:",smoke_score)

### 6. Extract and cache dev activations

Cache identity includes row IDs, state hashes, policy questions, model and code revisions, precision and sampled layers. Changing these invalidates the cache. Caches contain activations, not model weights. Oversize inputs raise an error rather than quietly changing the evaluation set.

In [ ]:
def extract_frame(df, name, requested_layers=LAYERS, variant="original"):
    identity = dict(ids=df.id.tolist(), hashes=df.state_hash.tolist(), questions=QSET,
        model=MODEL_ID, model_rev=MODEL_REV, code_rev=KEV_CODE_REV,
        benchmark_rev=BENCHMARK_REV, dtype=str(DTYPE), base_rev=checkpoint.meta.base_revision,
        layers=requested_layers, variant=variant, question_batch=QUESTION_BATCH,
        max_state_tokens=MAX_STATE_TOKENS,max_row_tokens=MAX_ROW_TOKENS,adapter_merged=False)
    cache_key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    path = OUT / f"{name}_{cache_key}.npz"
    if path.exists():
        z = np.load(path)
        return ({f: {l: z[f"{f}_{l}"] for l in requested_layers} for f in FEATURES},
                z["native"], z["per_question"], z["tokens"])
    collected = {f: {l: [] for l in requested_layers} for f in FEATURES}
    native, per_question, tokens = [], [], []
    started = time.perf_counter()
    for state in tqdm(df.state, desc=name):
        feats, score, scores, length = predict_and_capture(state, requested_layers, variant)
        for f in FEATURES:
            for l in requested_layers:
                collected[f][l].append(feats[f][l])
        native.append(score); per_question.append(scores); tokens.append(length)
    acts = {f: {l: np.stack(collected[f][l]) for l in requested_layers} for f in FEATURES}
    payload = {f"{f}_{l}": acts[f][l] for f in FEATURES for l in requested_layers}
    np.savez_compressed(path, **payload, native=np.array(native),
        per_question=np.array(per_question), tokens=np.array(tokens))
    path.with_suffix(".json").write_text(json.dumps(identity, indent=2))
    print(f"{name}: {len(df)} rows in {(time.perf_counter()-started)/60:.1f} minutes;",
          "longest question sequence:", max(tokens))
    return acts, np.array(native), np.array(per_question), np.array(tokens)

A_train, p_train_native, _, _ = extract_frame(train, "train")
A_valid, p_valid_native, _, _ = extract_frame(valid, "validation")
y_train, y_valid = train.y.to_numpy(), valid.y.to_numpy()

### 7. Train linear probes; select all settings on validation

Difference-of-means uses your midpoint construction. Its score is a margin, not a calibrated probability. LR uses training-only standardization. Compare against native scores tuned on the same validation set and against a character n-gram text classifier trained on the same labels.

A small search and modest dev split make this exploratory. If the validation curve is noisy, gather more independently labelled training data rather than repeatedly adjusting settings after seeing test.

In [ ]:
class MeanProbe:
    def fit(self, X, y):
        positive, negative = X[y==1].mean(0), X[y==0].mean(0)
        self.direction = positive-negative
        self.direction /= max(np.linalg.norm(self.direction), 1e-12)
        self.midpoint = (positive+negative)/2
        return self
    def decision_function(self, X):
        return (X-self.midpoint) @ self.direction


def choose_threshold(y, scores, probability=False):
    # Maximise validation balanced accuracy; include the meaningful default.
    default = .5 if probability else 0.
    candidates = np.unique(np.r_[np.quantile(scores, np.linspace(0,1,51)),
        default, np.nextafter(float(np.max(scores)), np.inf)])
    values = np.array([balanced_accuracy_score(y, scores >= t) for t in candidates])
    tied = np.flatnonzero(values == values.max())
    i = min(tied, key=lambda j: abs(candidates[j]-default))
    return float(candidates[i]), float(values[i])

search = []
fitted = {}
for feature in FEATURES:
    for layer in LAYERS:
        Xtr, Xva = A_train[feature][layer], A_valid[feature][layer]
        candidates = [("MM", None, MeanProbe().fit(Xtr, y_train))]
        for C in [.001, .01, .1]:
            lr = make_pipeline(StandardScaler(), LogisticRegression(
                C=C, max_iter=2500, solver="liblinear", random_state=SEED))
            lr.fit(Xtr, y_train)
            candidates.append(("LR", C, lr))
        for kind, C, probe in candidates:
            score = probe.decision_function(Xva)
            threshold, ba = choose_threshold(y_valid, score)
            key = (feature, layer, kind, C)
            fitted[key] = probe
            search.append(dict(feature=feature, layer=layer, kind=kind, C=C,
                threshold=threshold, validation_BA=ba,
                validation_AUROC=roc_auc_score(y_valid, score), key=key))
search_df = pd.DataFrame(search)
search_df.to_csv(OUT / "validation_search.csv", index=False)

def select_best(kind):
    return search_df[search_df.kind==kind].sort_values(
        ["validation_AUROC", "validation_BA", "layer"], ascending=[False, False, True]).iloc[0]
BEST = {kind: select_best(kind) for kind in ["MM","LR"]}
NATIVE_THRESHOLD, _ = choose_threshold(y_valid, p_valid_native, probability=True)
TEXT_CANDIDATES = []
for C in [.1, 1., 10.]:
    clf = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(3,5),
        min_df=1, max_features=20000), LogisticRegression(C=C, max_iter=2500,
        solver="liblinear", random_state=SEED))
    clf.fit([r["text"] for r in train.state], y_train)
    vs = clf.predict_proba([r["text"] for r in valid.state])[:,1]
    threshold, ba = choose_threshold(y_valid, vs, probability=True)
    TEXT_CANDIDATES.append((roc_auc_score(y_valid,vs), ba, C, threshold, clf))
_, _, TEXT_C, TEXT_THRESHOLD, TEXT_MODEL = max(TEXT_CANDIDATES, key=lambda a:(a[0],a[1]))
print("Frozen choices:")
for kind, choice in BEST.items():
    print(kind, choice[["feature","layer","C","threshold","validation_AUROC"]].to_dict())
print("Native threshold:", NATIVE_THRESHOLD, "| text C / threshold:", TEXT_C, TEXT_THRESHOLD)

plt.figure(figsize=(9,4))
for feature in FEATURES:
    for kind in ["MM","LR"]:
        # LR curve shows its validation-selected C at each layer; diagnostic, not test results.
        sub = search_df[(search_df.feature==feature)&(search_df.kind==kind)]
        curve = sub.groupby("layer").validation_AUROC.max()
        plt.plot(curve.index, curve.values, marker="o", label=f"{kind}: {feature}")
plt.axhline(roc_auc_score(y_valid,p_valid_native), color="black", ls="--", label="Native score")
plt.axhline(.5,color="gray",ls=":")
plt.xlabel("Decoder block (zero-based)"); plt.ylabel("Validation AUROC")
plt.title("Select the layer on validation, before test"); plt.legend(fontsize=8)
savefig("01_validation_layer_sweep")

### 8. Freeze choices, then open test

Native max-over-questions is not generally a calibrated probability of *any* policy violation. Its Brier score is a score diagnostic, not proof of calibration. MM and LR margins are evaluated with AUROC and selected decision thresholds; no probability-calibration claim is made for them.

In [ ]:
frozen = {kind: {k: (None if isinstance(v, (float, np.floating)) and np.isnan(v)
    else v.item() if isinstance(v, np.generic) else v)
    for k,v in row.to_dict().items() if k != "key"} for kind,row in BEST.items()}
(OUT / "frozen_selection.json").write_text(json.dumps(dict(probes=frozen,
    native_threshold=NATIVE_THRESHOLD, text_C=TEXT_C, text_threshold=TEXT_THRESHOLD), indent=2))

test = dedupe(load_rows("test"))
before = len(test)
# Exclude any exact input or related group appearing anywhere in the dev split.
test = test[~test.group.isin(dev.group) & ~test.state_hash.isin(dev.state_hash)].reset_index(drop=True)
print("Excluded dev/test overlapping rows:", before-len(test))
if SMOKE:
    test = small_group_sample(test, 12)
assert test.y.nunique() == 2
assert set(test.group).isdisjoint(set(dev.group))
NEEDED_LAYERS = sorted({int(row.layer) for row in BEST.values()} | {TRANSFER_LAYER})
A_test, p_test_native, q_test_native, test_tokens = extract_frame(test, "test", NEEDED_LAYERS)
y_test = test.y.to_numpy()
test_scores = {"Native @0.5": p_test_native, "Native tuned": p_test_native}
thresholds = {"Native @0.5": .5, "Native tuned": NATIVE_THRESHOLD}
for kind, choice in BEST.items():
    test_scores[kind] = fitted[choice.key].decision_function(A_test[choice.feature][int(choice.layer)])
    thresholds[kind] = float(choice.threshold)
test_scores["Text n-gram"] = TEXT_MODEL.predict_proba([r["text"] for r in test.state])[:,1]
thresholds["Text n-gram"] = TEXT_THRESHOLD
# Negative control: shuffle training labels, keep the selected LR layer/pooling/C fixed.
lr_choice = BEST["LR"]
shuffled = make_pipeline(StandardScaler(), LogisticRegression(C=float(lr_choice.C),
    max_iter=2500, solver="liblinear", random_state=SEED))
shuffled.fit(A_train[lr_choice.feature][int(lr_choice.layer)],
             np.random.default_rng(SEED).permutation(y_train))
test_scores["Shuffled-label LR"] = shuffled.decision_function(A_test[lr_choice.feature][int(lr_choice.layer)])
thresholds["Shuffled-label LR"] = 0.
# One shuffled run is a sanity check, not a full permutation significance test.

def metric_values(y, scores, threshold):
    pred = scores >= threshold
    return dict(AUROC=roc_auc_score(y,scores), AP=average_precision_score(y,scores),
        accuracy=accuracy_score(y,pred), balanced_accuracy=balanced_accuracy_score(y,pred),
        catch_rate=float(pred[y==1].mean()), false_block_rate=float(pred[y==0].mean()))

report = pd.DataFrame([dict(model=name, **metric_values(y_test,s,thresholds[name]))
                       for name,s in test_scores.items()]).set_index("model")
display(report.round(3)); report.to_csv(OUT / "test_metrics.csv")
print("Test rows:",len(test),"| groups:",test.group.nunique(),"| label prevalence:",y_test.mean())
print("These are local readable-topic results, not the published all-suite leaderboard.")

### 9. Test graphs and uncertainty

Bootstrap whole groups, preserving related safe / blocked pairs. The intervals condition on the already selected probe; they do not capture uncertainty from training-set sampling or the small validation search. Use the paired difference interval to assess evidence for an improvement over a tuned native score.

In [ ]:
def group_bootstrap_indices(df, n_boot=N_BOOT):
    rng = np.random.default_rng(SEED)
    groups = df.group.unique()
    indices = {g:np.flatnonzero(df.group.to_numpy()==g) for g in groups}
    for _ in range(n_boot):
        take = np.concatenate([indices[g] for g in rng.choice(groups,len(groups),replace=True)])
        if df.y.iloc[take].nunique()==2:
            yield take
boot = list(group_bootstrap_indices(test))
intervals = {}
for name,s in test_scores.items():
    samples = [balanced_accuracy_score(y_test[i],s[i]>=thresholds[name]) for i in boot]
    intervals[name] = np.quantile(samples,[.025,.975])
summary = report.copy()
summary["BA_low"] = [intervals[n][0] for n in summary.index]
summary["BA_high"] = [intervals[n][1] for n in summary.index]
summary.to_csv(OUT / "test_metrics_with_intervals.csv")

fig, axes = plt.subplots(1,2,figsize=(13,4.4))
for name,s in test_scores.items():
    if name=="Native tuned": continue  # Same ROC as the native default threshold.
    fpr,tpr,_=roc_curve(y_test,s)
    axes[0].plot(fpr,tpr,label=f"{name}: {roc_auc_score(y_test,s):.3f}")
axes[0].plot([0,1],[0,1],"k:"); axes[0].set(xlabel="False-block rate",ylabel="Catch rate",title="Held-out ROC")
axes[0].legend(fontsize=8)
names=list(summary.index); means=summary.balanced_accuracy.to_numpy()
lo=np.array([intervals[n][0] for n in names]); hi=np.array([intervals[n][1] for n in names])
# An empirical percentile interval can exclude its point estimate; draw endpoints directly.
axes[1].barh(names,means,color="#457b9d")
axes[1].hlines(np.arange(len(names)),lo,hi,color="black",linewidth=2)
axes[1].scatter(means,np.arange(len(names)),color="white",edgecolor="black",zorder=3)
axes[1].axvline(.5,color="gray",ls=":"); axes[1].set(xlim=(0,1),xlabel="Balanced accuracy",title="95% group-bootstrap intervals")
savefig("02_test_roc_and_accuracy")

fig, axes=plt.subplots(1,3,figsize=(12,3.8))
for ax,name in zip(axes,["Native @0.5","Native tuned","LR"]):
    cm=confusion_matrix(y_test,test_scores[name]>=thresholds[name],labels=[0,1])
    ax.imshow(cm,cmap="Blues")
    for i in range(2):
        for j in range(2):ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=16)
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Allow","Block"],yticklabels=["Allow","Block"],
           xlabel="Prediction",ylabel="Reference label",title=name)
savefig("03_test_confusion_matrices")

c=BEST["LR"]; Xtr=A_train[c.feature][int(c.layer)]; Xte=A_test[c.feature][int(c.layer)]
pca=PCA(n_components=2,svd_solver="randomized",random_state=SEED).fit(Xtr)
Z=pca.transform(Xte)
plt.figure(figsize=(7,5))
for label,color,text in [(0,"#457b9d","Allow"),(1,"#e76f51","Block")]:
    ix=y_test==label; plt.scatter(Z[ix,0],Z[ix,1],c=color,s=20,alpha=.65,label=text)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} train variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} train variance)")
plt.title(f"Test projection; PCA fitted on train only, block {int(c.layer)}");plt.legend()
savefig("04_train_fitted_pca")
# PCA separation alone is not the test of linear decodability; use the held-out probe metrics.

differences={}
for kind in ["MM","LR"]:
    ds=[balanced_accuracy_score(y_test[i],test_scores[kind][i]>=thresholds[kind])-
        balanced_accuracy_score(y_test[i],p_test_native[i]>=NATIVE_THRESHOLD) for i in boot]
    point=report.loc[kind,"balanced_accuracy"]-report.loc["Native tuned","balanced_accuracy"]
    low,high=np.quantile(ds,[.025,.975])
    differences[kind]=dict(BA_difference=float(point),low=float(low),high=float(high))
display(pd.DataFrame(differences).T.round(3))
(OUT/"paired_probe_minus_tuned_native.json").write_text(json.dumps(differences,indent=2))

### 10. Breakdown and example errors

These tables can suggest failure modes, but do not use them to revise this run's probe or threshold. Any revision requires a new untouched test set. Labels in this suite were AI drafted and cross-checked; inspect ambiguous examples with the written policy before interpreting disagreement as a model failure.

In [ ]:
topic_rows=[]
for topic,sub in test.groupby("topic"):
    idx=sub.index.to_numpy(); y=y_test[idx]
    if len(np.unique(y))<2:continue
    for name in ["Native @0.5","Native tuned","LR","Text n-gram"]:
        topic_rows.append(dict(topic=topic,model=name,n=len(idx),
            **metric_values(y,test_scores[name][idx],thresholds[name])))
topic_metrics=pd.DataFrame(topic_rows)
topic_metrics.to_csv(OUT/"per_topic_metrics.csv",index=False)
if len(topic_metrics):
    table=topic_metrics.pivot(index="topic",columns="model",values="balanced_accuracy")
    table.plot.barh(figsize=(10,6),xlim=(0,1));plt.xlabel("Balanced accuracy")
    plt.title("Same policy set; per-topic test examples");savefig("05_per_topic_accuracy")
error_table=pd.DataFrame(dict(id=test.id,group=test.group,topic=test.topic,label=y_test,
    native_score=p_test_native,lr_margin=test_scores["LR"],
    native_block=p_test_native>=.5,lr_block=test_scores["LR"]>=thresholds["LR"],
    text=[s["text"] for s in test.state]))
error_table.to_csv(OUT/"test_predictions.csv",index=False)
display(error_table[(error_table.native_block!=error_table.label)|
                    (error_table.lr_block!=error_table.label)].head(10))

### 11. Optional stability tests (no refitting)

Reverse JSON field order and reverse the two options while preserving their descriptions and true/false interpretation. Labels remain unchanged. The subset is chosen by seeded group sampling, not by errors. Both the native head and frozen LR probe see the modified inputs.

A low flip rate is useful only alongside clean accuracy: a consistently wrong classifier is stable too. A probability change and a probe-margin change have different units. We plot decision flips and balanced accuracy, not a misleading shared score scale.

In [ ]:
robust_rows=[]
if RUN_ROBUSTNESS:
    subset=small_group_sample(test,6 if SMOKE else 20)
    original_positions=np.array([test.index[test.id==row_id][0] for row_id in subset.id])
    c=BEST["LR"]; layer=int(c.layer)
    for variant in ["json_reorder","option_reorder"]:
        av,pv,_,_=extract_frame(subset,variant,[layer],variant)
        lv=fitted[c.key].decision_function(av[c.feature][layer])
        for name,changed in [("Native @0.5",pv),("Native tuned",pv),("LR",lv)]:
            original=test_scores[name][original_positions]
            threshold=thresholds[name]
            robust_rows.append(dict(variant=variant,model=name,n=len(subset),
                flip_rate=float(np.mean((changed>=threshold)!=(original>=threshold))),
                clean_BA=balanced_accuracy_score(subset.y,original>=threshold),
                changed_BA=balanced_accuracy_score(subset.y,changed>=threshold),
                mean_abs_score_change=float(np.mean(abs(changed-original)))))
    robustness=pd.DataFrame(robust_rows)
    display(robustness.round(3));robustness.to_csv(OUT/"robustness.csv",index=False)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    robustness.pivot(index="variant",columns="model",values="flip_rate").plot.bar(ax=axes[0])
    axes[0].set(ylabel="Decision flip fraction",title="Semantic-preserving order changes",ylim=(0,1))
    robustness.pivot(index="variant",columns="model",values="changed_BA").plot.bar(ax=axes[1])
    axes[1].set(ylabel="Balanced accuracy after change",title="Frozen thresholds and probes",ylim=(0,1))
    savefig("06_order_robustness")

### 12. Optional transfer test: leave two topic families out of probe training

This is predeclared, not chosen because a topic scored badly. Uses fixed answer-mean features, the midpoint sampled block, and C=0.01; these settings are not inherited from the earlier all-topic selection. Refits a probe with MedicalDiagnosis and LegalAdvice examples removed from training/validation, then evaluates them separately. All nine policies remain in the prompt. This measures transfer to **unseen example families under known policies**, not unseen policy definitions. The existing test set has already been viewed, so this is a secondary, exploratory analysis.

In [ ]:
HELD_OUT_TOPICS={"MedicalDiagnosis","LegalAdvice"}
keep_train=~train.topic.isin(HELD_OUT_TOPICS).to_numpy()
keep_valid=~valid.topic.isin(HELD_OUT_TOPICS).to_numpy()
keep_test=test.topic.isin(HELD_OUT_TOPICS).to_numpy()
transfer_rows=[]
if all(len(np.unique(y))==2 for y in [y_train[keep_train],y_valid[keep_valid],
                                    y_test[keep_test],y_test[~keep_test]]):
    feature="answer_mean"; layer=TRANSFER_LAYER
    transfer=make_pipeline(StandardScaler(),LogisticRegression(C=.01,
        max_iter=2500,solver="liblinear",random_state=SEED))
    transfer.fit(A_train[feature][layer][keep_train],y_train[keep_train])
    vt=transfer.decision_function(A_valid[feature][layer][keep_valid])
    tt,_=choose_threshold(y_valid[keep_valid],vt)
    ts=transfer.decision_function(A_test[feature][layer])
    nt,_=choose_threshold(y_valid[keep_valid],p_valid_native[keep_valid],probability=True)
    for name,s,t in [("Transfer LR",ts,tt),("Native tuned on seen topics",p_test_native,nt)]:
        for tag,mask in [("Seen topic families",~keep_test),("Held-out topic families",keep_test)]:
            transfer_rows.append(dict(model=name,subset=tag,n=int(mask.sum()),
                **metric_values(y_test[mask],s[mask],t)))
    transfer_table=pd.DataFrame(transfer_rows)
    display(transfer_table.round(3));transfer_table.to_csv(OUT/"topic_transfer.csv",index=False)
    transfer_table.pivot(index="subset",columns="model",values="balanced_accuracy").plot.bar(
        figsize=(8,4),ylim=(0,1));plt.ylabel("Balanced accuracy");plt.title("Exploratory topic-family transfer")
    savefig("07_topic_family_transfer")
else:
    print("Transfer test skipped: this smoke subset lacks both labels in a required slice.")

### 13. Export

Download the ZIP before disconnecting Colab. It includes graphs, activation caches, settings and result tables. For another model, replace the loader, native rendering and position capture together; the analysis can be reused. Changing only `MODEL_ID` is not a valid cross-model experiment.

**Interpretation:**
- Probe beats native @0.5 but not native tuned: threshold choice may explain the gain.
- Probe beats tuned native and text baseline, with a positive paired interval: evidence for useful learned readout information on this dataset.
- Probe loses on held-out topic families or order changes: limited generalization / robustness.
- High accuracy means information is decodable; it does **not** prove the model uses a specific feature causally, nor that its head alone caused its errors.

To test the stronger claim that internal representations understand the policy while the head fails, add controlled policy changes, matched keyword controls, and activation interventions on independent data. Do not infer anything about closed Jev's internal states from another model's probes.

A probe still requires the underlying forward pass. This experiment does not establish lower inference cost or a production-ready replacement for Bedrock Guardrails.

In [ ]:
manifest=dict(seed=SEED,smoke=SMOKE,model=MODEL_ID,model_revision=MODEL_REV,
    base=checkpoint.meta.base,base_revision=checkpoint.meta.base_revision,
    kev_code_revision=KEV_CODE_REV,benchmark_code_revision=BENCHMARK_REV,
    data=DATA_ID,data_revision=DATA_REV,precision=str(DTYPE),
    head_temperature=float(model.head.temperature),adapter_merged=False,
    max_state_tokens=MAX_STATE_TOKENS,max_row_tokens=MAX_ROW_TOKENS,
    gpu=torch.cuda.get_device_name(0),python=platform.python_version(),
    question_keys=QKEYS,layers=LAYERS,question_batch=QUESTION_BATCH,
    row_counts={"train":len(train),"validation":len(valid),"test":len(test)},
    dataset_audit=row_audit,bootstrap=N_BOOT,
    packages={p:im.version(p) for p in ["torch","transformers","peft","kev",
        "scikit-learn","huggingface-hub","pandas","numpy","matplotlib"]})
(OUT/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
for name,df in [("train",train),("validation",valid),("test",test)]:
    df[["id","group","y","topic","source","state_hash"]].to_csv(OUT/f"{name}_row_manifest.csv",index=False)
archive=OUT.with_suffix(".zip")
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():z.write(path,arcname=path.relative_to(OUT))
print("Saved:",archive)
from google.colab import files
files.download(str(archive))